# 05 · Validation plan — kinetic assay, controls, directed evolution

**Standard slot:** *validation plan.* **For Project 18 this means:** turn the catalytic-geometry-
filtered set into a costed **kinetic-assay plan** with the right controls (incl. a catalytic-dead Ala
mutant) and a **directed-evolution** plan for hits (D4/D5).

This is the deliverable that states, plainly: **in-silico geometry is a hypothesis; the assay tests it.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the ranked filter (catalytic geometry first), capped at **< 96** so they
fit a single screening plate with controls. Diversity matters — don't pick 96 near-identical designs.

In [ ]:
import pandas as pd
try:
    ranked = pd.read_csv("results/ranked.csv")
except FileNotFoundError:
    ranked = pd.read_csv("results/campaign.csv")

# Prefer designs passing the catalytic-geometry bar; cap under 96 (leave wells for controls).
ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
selected = ok.head(88).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds; record why each was chosen in your report.")

## 2 · The kinetic assay (UV product readout)
- **Express + purify:** *E. coli* BL21(DE3), 16–18 °C overnight; His-tag → IMAC → SEC polish.
- **Assay:** follow **product** formation (2-hydroxy-5-nitrobenzonitrile / nitrophenolate) by
  absorbance at its λmax in a plate reader; take initial rates across substrate concentrations.
- **Kinetics:** fit **kcat** and **KM** (and kcat/KM) from a Michaelis–Menten / linear-regime fit.
- **Readout note:** background (uncatalysed) Kemp elimination is non-zero — subtract the buffer-only
  rate, and account for it in every well.

In [ ]:
controls = {
    "POSITIVE — natural/reference Kemp eliminase": "a verified KE/HG-series enzyme; confirms the assay works",
    "NEGATIVE — catalytic base -> Ala 'dead' mutant": "SAME design, base mutated to Ala; cleanest negative — "
        "loss of activity pins catalysis to that residue",
    "NEGATIVE — heat-killed enzyme": "boiled aliquot; rules out non-protein / contaminant rates",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background activity",
    "BLANK — buffer + substrate only": "the uncatalysed background rate to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")

## 3 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
and assay reagents at your institution's rates; the cell prints a template to fill in your report.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis", "1 plate", "fill"),
    ("IMAC purification (plate format)", "1 plate", "fill"),
    ("5-nitrobenzisoxazole substrate", "stock", "fill"),
    ("Plate-reader time (kinetics)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:52s} {scale:12s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+assay 1-2 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here, "
      "but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 4 · Directed-evolution plan for hits `[stretch]`
The honest history of Kemp eliminases: the first designs were weak and only became efficient after
**directed evolution** (e.g. the HG3 → HG3.17 lineage). For any hit:
- **Libraries:** saturation/site-saturation around the catalytic base, the π-stack, the H-bond donor,
  and the **second shell** (residues tuning the base's pKa and the pocket).
- **Selection/screen:** the **same UV assay** as the readout — screen in plates, pick fast wells,
  sequence winners, recombine beneficial mutations over rounds.
- **Stop criteria:** target kcat/KM, plateau across rounds, or budget; report the trajectory honestly.

In [ ]:
print("Directed-evolution loop (stretch): mutate active-site/second-shell -> express -> UV screen ->")
print("sequence top wells -> recombine -> repeat. Cite the HG3.17 / Arnold directed-evolution context.")
print("Reminder for the thesis: report the hit rate and the kcat/KM TRAJECTORY, not just the best clone.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, catalytic-geometry-passing designs.
- [ ] Kinetic-assay plan: UV product readout, kcat/KM, with **all** controls (catalytic-dead Ala
      mutant, heat-killed, empty vector, blank), costed + timed.
- [ ] Directed-evolution plan for hits `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ activity" stated plainly.

You're done — and Projects 19–21, 24 reuse this theozyme→scaffold→sequence→geometry template.